# Qual é o melhor plano?

Você trabalha como analista para a empresa de telecomunicações Megaline. A empresa oferece aos clientes dois planos pré-pagos: Surf e Ultimate. O departamento comercial quer saber qual dos planos gera mais receita para ajustar o orçamento de publicidade.

Você vai realizar uma análise preliminar dos planos com base em uma pequena seleção de clientes. Você terá dados de 500 clientes da Megaline: que clientes são, de onde eles são, qual plano usam e o número de chamadas e mensagens realizadas em 2018. Seu trabalho é analisar o comportamento dos clientes e determinar qual plano pré-pago gera mais receita.

[Fornecemos alguns comentários para guiar sua linha de raciocínio enquanto você trabalha neste projeto. Entretanto, certifique-se de remover todos os comentários entre colchetes antes de enviar o projeto.]

[Antes de começar a análise dos dados, explique com suas próprias palavras o propósito do projeto e as ações que planeja realizar.]

[Tenha em mente que estudar, modificar e analisar dados é um processo iterativo. É normal retornar a etapas anteriores e corrigir/expandir algo para permitir as próximas etapas.]

## Inicialização

In [1]:
# Carregando todas as bibliotecas
import pandas as pd
from scipy import stats as st
import numpy as np
from matplotlib import pyplot as plt


ModuleNotFoundError: No module named 'matplotlib'

## Carregue os dados

In [ ]:
# Carregue os arquivos de dados em diferentes DataFrames
df_calls = pd.read_csv('/datasets/megaline_calls.csv')
df_internet = pd.read_csv('/datasets/megaline_internet.csv')
df_messages = pd.read_csv('/datasets/megaline_messages.csv')
df_plans = pd.read_csv('/datasets/megaline_plans.csv')
df_users = pd.read_csv('/datasets/megaline_users.csv')

## Prepare os dados

## Planos

In [ ]:
# Imprima informações gerais/resumo sobre o DataFrame dos planos
print(df_plans.info())


* df_plans contém 8 colunas e duas linhas
* não há nenhum valor explicitamente nulo ou ausente
* cabeçalho contém nomes coerente e não precisa de correção
* os tipos de variáveis parecem adequados aos valores que elas representam


In [ ]:

# Imprima uma amostra de dados dos planos
print(df_plans.head())


df_plans é uma tabela simples e coerente que apresenta as características dos planos disponibilizados e não precisa de ajustes.
Suas características são:

* contém 8 colunas e 2 linhas
* não há nenhum valor explicitamente nulo ou ausente
* não há valor duplicado
* cabeçalho contém nomes coerente e não precisa de correção
* os tipos de variáveis parecem adequados aos valores que elas representam


## Corrija os dados

Até o presente momento, não foram encontrados dados que precisacem ser corrigidos.

## Enriqueça os dados

Até o presente momento, não foi concebida necessidade de enriquecimento dos dados do df_plans.

## Usuários

In [ ]:
# Imprima informações gerais/resumo sobre o DataFrame dos usuários
print(df_users.info())


* contém 8 colunas e 500 linhas
* cabeçalho contém nomes coerente e não precisa de correção
* os tipos de variáveis das colunas 'reg_date' e ' churn_date' precisam ser avaliados se seus tipos estão o mais coerente com o contexto da tabela.
* aparentemente, não há nenhum valor explicitamente nulo, mas é necessário investigar a coluna 'churn_date' para entender se os valores ausentes estão dentro do contexto esperado.
* é necessário investigar a presença de dados duplicados

In [ ]:
# Imprima uma amostra de dados dos usuários
print(df_users.head(5))

In [ ]:
# Encontrando linhas não nulas na coluna 'churn_date' para avaliação
df_users_churn = df_users[~df_users['churn_date'].isna()]
print(df_users_churn.head(1))

* os tipos de variáveis das colunas 'reg_date' e ' churn_date' precisam ajustar seus tipos para datetime no intuito de facilitar a manipulação dos dados
* é possível aprimorar estas colunas de datas separando por seus componentes dia, mês e ano
* o conteúdo da coluna 'city' está mais extenso do que o que se propõe, é
* é possível aprimorar esta coluna dividindo em duas partes, uma com a MSA (Metropolitan Statistical Area) que ela abrange e a outra com o nome das cidades desta 'área estatística metropolitana'

##### RESUMO DA TABELA df_users

df_users é uma tabela que apresenta a amostra dos clientes que serão estudados.
Suas características básicas são:
* contém 8 colunas e 500 linhas
* cabeçalho contém nomes coerente e não precisa de correção
* aparentemente, não há nenhum valor explicitamente nulo, a coluna 'churn_date' contém valores ausentes, mas estão dentro do contexto esperado.

Ela precisa das seguintes intervenções:
* os tipos de variáveis das colunas 'reg_date' e ' churn_date' precisam ajustar seus tipos para datetime no intuito de facilitar a manipulação dos dados
* é necessário investigar a presença de dados duplicados

Ela deve ser aprimorada da seguinte forma:
* o conteúdo da coluna 'city' está mais extenso do que o que se propõe, é possível aprimorar esta coluna dividindo em duas partes, uma com a MSA (Metropolitan Statistical Area) que ela abrange e a outra com o nome das cidades desta 'área estatística metropolitana
* as colunas 'reg_date' e ' churn_date' terão suas datas separando por seus componentes dia, mês e ano
* os valores ausentes serão substituídos por 0

### Corrija os dados

In [ ]:
# ajustando as colunas 'reg_date' e 'churn_date' para datetime
df_users['reg_date'] = pd.to_datetime(df_users['reg_date'], format='%Y-%m-%d')
df_users['churn_date'] = pd.to_datetime(df_users['churn_date'], format='%Y-%m-%d')

print(df_users.info())

* coluna 'reg_date' e 'churn_date' que contém conteúdo de data foram modificadas para o tipo datetime

In [ ]:
# Investigando valores duplicados
print('Quantidade de linhas explicitamente duplicadas:', df_users.duplicated().sum())

* não foram encontradas linhas duplicadas

### Enriqueça os dados

In [ ]:
# dividindo a coluna 'city' em 'city_msa' e 'state_msa'
df_users[['city_msa','state_msa']] = df_users['city'].str.split(',', n=1, expand=True)

# limpando espaços em branco e removendo o sufixo 'MSA'
df_users['state_msa'] = df_users['state_msa'].str.strip()
df_users['state_msa'] = df_users['state_msa'].str.replace('MSA', '', regex=False).str.strip()

# apagando a coluna original 'city'
df_users = df_users.drop(columns='city')

In [ ]:
# separando os componentes das colunas 'reg_date' e ' churn_date' em dia, mês e ano

# trabalhando na coluna 'reg_date'
df_users['reg_day'] = df_users['reg_date'].dt.day 
df_users['reg_month'] = df_users['reg_date'].dt.month 
df_users['reg_year'] = df_users['reg_date'].dt.year

df_users = df_users.drop(columns='reg_date')              # apagando a coluna 'reg_date' original

# trabalhando na coluna 'churn_date'
df_users['churn_day'] = df_users['churn_date'].dt.day 
df_users['churn_month'] = df_users['churn_date'].dt.month 
df_users['churn_year'] = df_users['churn_date'].dt.year

df_users = df_users.drop(columns='churn_date')              # apagando a coluna 'churn_date' original

# imprimindo para garantir que o objetivo foi alcançado
print(df_users.head())

In [ ]:
# substituindo os valores ausentes por 0
df_users['churn_day'] = df_users['churn_day'].fillna(0).astype(int)
df_users['churn_month'] = df_users['churn_month'].fillna(0).astype(int)
df_users['churn_year'] = df_users['churn_year'].fillna(0).astype(int)

print(df_users.head())



* a coluna 'city' foi dividida e substituída pelas seguintes colunas:

- 'city_msa' que contém o nome explícito das cidades da MSA;

- 'state_msa' que contém o estado da MSA


* as colunas com datas tiveram seus componentes separados em dia, mês e ano 

## Chamadas

In [ ]:
# Imprima informações gerais/resumo sobre o DataFrame das chamadas
print(df_calls.info())


* contém 4 colunas e 137735 linhas
* cabeçalho contém nomes coerentes, mas a coluna 'duration' poderia especificar a unidade de medida em minutos
* aparentemente, não há nenhum valor explicitamente nulo
* é necessário investigar a presença de dados duplicados
* aparentemente, os tipos das variáveis não estão adequados, mas é necessário imprimir uma amostra para avaliar.


In [ ]:
# Imprima uma amostra de dados das chamadas
print(df_calls.head())


* a coluna 'call_date' precisa ter seu tipo modificado para datetime
* é possível aprimorar a coluna call_date separando por seus componentes dia, mês e ano

##### RESUMO DA TABELA df_calls

df_calls é uma tabela que apresenta os dados sobre as chamadas.
Suas características básicas são:
* contém 4 colunas e 137735 linhas
* cabeçalho contém nomes coerentes que não precisam de conserto
* aparentemente, não há nenhum valor explicitamente nulo
* os tipos das variáveis estão adequados, exceto na coluna 'call_date'. Contudo, a coluna 'duration' ficará em observação para avaliar a necessidade futura de modificar seus dados para datetime caso necessário.

Ela precisa das seguintes intervenções:
* a coluna 'call_date' precisa ter seu tipo modificado para datetime
* é necessário investigar a presença de dados duplicados

Ela deve ser aprimorada da seguinte forma:
* a coluna 'duration' deve especificar a unidade de medida em minutos
* a coluna 'call_date' terá sua data separada por seus componentes dia, mês e ano


### Corrija os dados

In [ ]:
# ajustando as colunas 'call_date' para datetime
df_calls['call_date'] = pd.to_datetime(df_calls['call_date'], format='%Y-%m-%d')

print(df_calls.info())


In [ ]:
# Investigando valores duplicados
print('Quantidade de linhas explicitamente duplicadas:', df_calls.duplicated().sum())


### Enriqueça os dados

In [ ]:
# modificar o nome da coluna 'duration' especificando o unidade de medida para 'duration_minutes'
df_calls = df_calls.rename(columns={'duration': 'duration_minutes'})

print(df_calls.head())


In [ ]:
# separando os componentes da coluna 'call_date' dia, mês e ano
df_calls['call_day'] = df_calls['call_date'].dt.day 
df_calls['call_month'] = df_calls['call_date'].dt.month 
df_calls['call_year'] = df_calls['call_date'].dt.year

df_calls = df_calls.drop(columns='call_date')              # apagando a coluna 'call_date' original

# imprimindo para garantir que o objetivo foi alcançado
print(df_calls.head())


* a coluna call_date foi modificada para tipo datetime
* a coluna call_date teve seus componentes separados em dia, mês e ano 
* não há valores explicitamente duplicados
* o cabeçalho da coluna 'duration' foi modificado para 'duration_minutes'


## Mensagens

In [ ]:
# Imprima informações gerais/resumo sobre o DataFrame das mensagens
print(df_messages.info())


* contém 3 colunas e 76051 linhas
* cabeçalho contém nomes coerentes que não precisam de ajuste
* aparentemente, não há nenhum valor explicitamente nulo
* é necessário investigar a presença de dados duplicados
* aparentemente, os tipos das variáveis não estão adequados, mas é necessário imprimir uma amostra para avaliar.


In [ ]:
# Imprima uma amostra dos dados das mensagens
print(df_messages.head())


* a coluna 'message_date' precisa ter seu tipo modificado para datetime
* é possível aprimorar a coluna message_date separando por seus componentes dia, mês e ano

#### RESUMO DA TABELA df_messages

df_messages é uma tabela que apresenta os dados sobre as mensagens de texto.
Suas características básicas são:
* contém 3 colunas e 76051 linhas
* cabeçalho contém nomes coerentes que não precisam de ajuste
* aparentemente, não há nenhum valor explicitamente nulo
* é necessário investigar a presença de dados duplicados
* os tipos das variáveis estão adequados, exceto na coluna 'message_date'.

Ela precisa das seguintes intervenções:
* a coluna 'message_date' precisa ter seu tipo modificado para datetime
* é necessário investigar a presença de dados duplicados

Ela deve ser aprimorada da seguinte forma:
* a coluna 'message_date' terá sua data separada por seus componentes dia, mês e ano


### Corrija os dados

In [ ]:
# ajustando as colunas 'message_date' para datetime
df_messages['message_date'] = pd.to_datetime(df_messages['message_date'], format='%Y-%m-%d')

print(df_messages.info())


In [ ]:
# Investigando valores duplicados
print('Quantidade de linhas explicitamente duplicadas:', df_messages.duplicated().sum())


### Enriqueça os dados

In [ ]:
# separando os componentes da coluna 'message_date' em dia, mês e ano
df_messages['message_day'] = df_messages['message_date'].dt.day 
df_messages['message_month'] = df_messages['message_date'].dt.month 
df_messages['message_year'] = df_messages['message_date'].dt.year

df_messages = df_messages.drop(columns='message_date')              # apagando a coluna 'message_date' original

# imprimindo para garantir que o objetivo foi alcançado
print(df_messages.head())


* a coluna 'message_date' foi modificada para tipo datetime
* não há valores explicitamente duplicados
* a coluna 'message_date' teve seus componentes separados em dia, mês e ano 


## Internet

In [ ]:
# Imprima informações gerais/resumo sobre o DataFrame da internet
print(df_internet.info())


* contém 4 colunas e 104.825 linhas
* cabeçalho contém nomes coerentes que não precisam de ajuste
* aparentemente, não há nenhum valor explicitamente nulo
* é necessário investigar a presença de dados duplicados
* aparentemente, os tipos das variáveis não estão adequados, mas é necessário imprimir uma amostra para avaliar.


In [ ]:
#  Imprima uma amostra de dados para o tráfego da internet
print(df_internet.head())


* a coluna 'session_date' precisa ter seu tipo modificado para datetime
* é possível aprimorar a coluna 'session_date' separando por seus componentes dia, mês e ano


#### RESUMO DA TABELA df_internet

df_internet é uma tabela que apresenta os dados sobre as sessões web.
Suas características básicas são:
* contém 4 colunas e 104.825 linhas
* cabeçalho contém nomes coerentes que não precisam de ajuste
* aparentemente, não há nenhum valor explicitamente nulo
* é necessário investigar a presença de dados duplicados
* aparentemente, os tipos das variáveis estão adequados, exceto na coluna'session_date'.

Ela precisa das seguintes intervenções:
* a coluna 'session_date' precisa ter seu tipo modificado para datetime
* é necessário investigar a presença de dados duplicados

Ela deve ser aprimorada da seguinte forma:
* a coluna 'session_date' terá sua data separada por seus componentes dia, mês e ano


### Corrija os dados

In [ ]:
# ajustando as colunas 'session_date' para datetime
df_internet['session_date'] = pd.to_datetime(df_internet['session_date'], format='%Y-%m-%d')

print(df_internet.info())


In [ ]:
# Investigando valores duplicados
print('Quantidade de linhas explicitamente duplicadas:', df_internet.duplicated().sum())


### Enriqueça os dados

In [ ]:
# separando os componentes da coluna 'call_date' em dia, mês e ano
df_internet['session_day'] = df_internet['session_date'].dt.day 
df_internet['session_month'] = df_internet['session_date'].dt.month 
df_internet['session_year'] = df_internet['session_date'].dt.year

df_internet = df_internet.drop(columns='session_date')              # apagando a coluna 'session_date' original

# imprimindo para garantir que o objetivo foi alcançado
print(df_internet.head())


* a coluna 'session_date' foi modificada para tipo datetime
* não há valores explicitamente duplicados
* a coluna 'session_date' teve seus componentes separados em dia, mês e ano 


## Estude as condições dos planos

[É fundamental entender como os planos funcionam, ou seja, como as cobranças dos usuários são feitas com base na assinatura. Sugerimos imprimir as informações sobre os planos para visualizar novamente as condições.]

In [ ]:
# Imprima as condições dos planos e certifique-se de que elas fazem sentido para você
print(df_plans.head())


## Agregue os dados por usuário

[Agora, como os dados estão limpos, os agregue por usuário e por período para ter apenas um registro dessas informações. Isso vai facilitar muito as próximas análises.]

In [ ]:
# Calcule o número de chamadas feitas por cada usuário por mês. Salve o resultado.

# Agregando os dados de número de chamadas feitas por cada usuário por mês
agg_num_calls = {'id': 'count'}
grp_calls = df_calls.groupby(['user_id','call_month'])
grp_calls.agg(agg_num_calls)

# Salvando em um df
df_num_calls = grp_calls.agg(agg_num_calls).reset_index().rename(columns={'id':'number_calls'})

# Criando uma coluna índice para unir df
df_num_calls['merge_key'] = df_num_calls['user_id'].astype(str) + '_' + df_num_calls['call_month'].astype(str)
print(df_num_calls)


In [ ]:
# Calcule a quantidade de minutos gastos por cada usuário por mês. Salve o resultado.

# Agregando a quantidade de minutos gastos por cada usuário por mês
agg_minutes = {'duration_minutes': 'sum'}
grp_calls = df_calls.groupby(['user_id','call_month'])
minutes_calls = grp_calls.agg(agg_minutes)

# Salvando em um df
df_minutes_calls = grp_calls.agg(agg_minutes).reset_index().rename(columns={'duration_minutes':'sum_duration_minutes'})

# Criando uma coluna índice para unir df
df_minutes_calls['merge_key'] = df_minutes_calls['user_id'].astype(str) + '_' + df_minutes_calls['call_month'].astype(str)
print(df_minutes_calls)


In [ ]:
# Calcule o número de mensagens enviadas por cada usuário por mês. Salve o resultado.

# Agregando o número de mensagens enviadas por cada usuário por mês
agg_message = {'id': 'count'}
grp_messages = df_messages.groupby(['user_id','message_month'])
number_messages = grp_messages.agg(agg_message)

# Salvando em um df
df_number_messages = grp_messages.agg(agg_message).reset_index().rename(columns={'id':'number_messages'})

# Criando uma coluna índice para unir df
df_number_messages['merge_key'] = df_number_messages['user_id'].astype(str) + '_' + df_number_messages['message_month'].astype(str)
print(df_number_messages)


In [ ]:
# Calcule o volume de tráfego de internet usado por cada usuário por mês. Salve o resultado.

# Agregando o volume de tráfego de internet usado por cada usuário por mês
agg_internet = {'mb_used': 'sum'}
grp_internet = df_internet.groupby(['user_id','session_month'])
mbs_used = grp_internet.agg(agg_internet)

# Salvando em um df
df_mbs_used = grp_internet.agg(agg_internet).reset_index().rename(columns={'mb_used':'sum_mb_used'})

# Criando uma coluna índice para unir df
df_mbs_used['merge_key'] = df_mbs_used['user_id'].astype(str) + '_' + df_mbs_used['session_month'].astype(str)
print(df_mbs_used)


In [ ]:
# Junte os dados de chamadas, minutos, mensagens e internet com base em user_id e month

# Juntando df com dados de chamadas e minutos
df_num_min = df_num_calls.merge(df_minutes_calls, on='merge_key', how='outer')
df_num_min = df_num_min.drop(columns=['user_id_x', 'user_id_y', 'call_month_x', 'call_month_y'])     # ajustando as colunas
#print(df_num_min)

# Juntando df com dados de chamadas, minutos e mensagens
df_num_min_mess = df_num_min.merge(df_number_messages, on='merge_key', how='outer')
df_num_min_mess = df_num_min_mess.drop(columns=['user_id', 'message_month'])                         
#print(df_num_min_mess)

# Juntando df com dados de chamadas, minutos, mensagens e internet
df_user_month = df_num_min_mess.merge(df_mbs_used, on='merge_key', how='outer')

# Ajustando a quantidade, tipo e ordem das colunas
df_user_month[['user_id','month_used']] = df_user_month['merge_key'].str.split('_', n=1, expand=True)
df_user_month = df_user_month.drop(columns=['session_month', 'merge_key'])
df_user_month['user_id'] = df_user_month['user_id'].astype(int)
df_user_month = df_user_month.sort_values('user_id')
df_user_month = df_user_month[['user_id', 'month_used', 'number_calls', 'sum_duration_minutes', 'number_messages', 'sum_mb_used']]

# Substituindo valores ausentes
df_user_month['number_calls'] = df_user_month['number_calls'].fillna(0).astype(int)
df_user_month['sum_duration_minutes'] = df_user_month['sum_duration_minutes'].fillna(0)
df_user_month['number_messages'] = df_user_month['number_messages'].fillna(0).astype(int)
df_user_month['sum_mb_used'] = df_user_month['sum_mb_used'].fillna(0)

print(df_user_month.tail(20))


In [ ]:
# ENTENDER COM SIMM:
# COMO 10 USER ID NÃO APARECEM NA TABELA df_user_month, MAS APARECEM NA TABELA df_month_plan_used
# qual a razão de ter aumentado o número de linha de 2254 para 2302 quando fiz o merge

#print(df_user_month['month_used'].value_counts(dropna=False))
#print(df_user_month[df_user_month['month_used'].isna()])
#print(df_user_month[df_user_month['user_id'] == 1473])

In [ ]:
# Adicione as informações sobre o plano
# criando um df resumido de usuários
df_users_fit = df_users[['user_id', 'plan', 'reg_month', 'churn_month']]

print(df_users_fit.tail())


In [ ]:
# Criando um df com o plano de cada usuário
df_users_plan = df_users_fit.merge(df_plans, 
                                   left_on='plan',
                                   right_on='plan_name').drop(columns='plan_name')
#print(df_users_plan.tail())


In [ ]:
# Criando um df com os dados do consumo mensal e quanto é cobrado por cada produto
df_month_plan_used = df_user_month.merge(df_users_plan, on='user_id', how='outer')

#print(df_month_plan_used.tail(20))


In [ ]:
# Removendo linhas ausentes após o merge

# NÃO APAGAR ATÉ MOSTRAR A SIMM

# print(df_month_plan_used['month_used'].value_counts(dropna=False))
# print(df_month_plan_used[df_month_plan_used['month_used'].isna()])

In [ ]:
print(df_plans)

In [ ]:
# Removendo linhas ausentes após o merge
df_month_plan_used = df_month_plan_used.dropna(subset=['month_used'])

#print(df_month_plan_used['month_used'].value_counts(dropna=False))

#print(df_month_plan_used.sample(10))

In [ ]:

# Criando uma nova coluna para calcular o custo extra por mensagem excedida
df_month_plan_used['usd_message_add'] = (
    (df_month_plan_used['number_messages']
     -df_month_plan_used['messages_included']).clip(lower=0) 
    * df_month_plan_used['usd_per_message']
)


In [ ]:
# Criando uma nova coluna para calcular o custo extra por minuto excedido
df_month_plan_used['usd_minute_add'] = (
    (df_month_plan_used['sum_duration_minutes']
     -df_month_plan_used['minutes_included']).clip(lower=0) 
    * df_month_plan_used['usd_per_minute']
)


In [ ]:
# Criando uma nova coluna para calcular o custo extra por internet excedida
df_month_plan_used['usd_mb_add'] = (
    (df_month_plan_used['sum_mb_used']
     -df_month_plan_used['mb_per_month_included']).clip(lower=0) 
    * (df_month_plan_used['usd_per_gb']/1024)
).round(2)

df_month_plan_used['month_used'] = df_month_plan_used['month_used'].astype(int)


In [ ]:
# Calcule a receita mensal para cada usuário
df_month_plan_used['monthly_revenue'] = (
    df_month_plan_used['usd_monthly_pay'] 
    + df_month_plan_used['usd_message_add'] 
    + df_month_plan_used['usd_minute_add'] 
    + df_month_plan_used['usd_mb_add']
).round(2)

print(df_month_plan_used.head())


## Estude o comportamento do usuário

[Calcule algumas estatísticas descritivas úteis para os dados agregados, o que costuma revelar uma imagem geral capturada pelos dados. Desenhe gráficos úteis para ajudar na compreensão. Já que a tarefa principal é comparar os planos e decidir qual é mais rentável, as estatísticas e os gráficos devem ser calculados por plano.]

[Existem dicas relevantes nos comentários para as chamadas. Essas dicas não foram fornecidas para as mensagens e internet, mas o princípio do estudo estatístico é o mesmo em todos os casos.]

### Chamadas

In [ ]:
# Implementando o df_plans com a coluna do plano por usuário

df_calls_plan = (df_calls.merge(df_users_fit[['user_id', 'plan']], 
                                         on='user_id', how='left').
                drop(columns=['call_year', 'call_day']))

print(df_calls_plan)


In [ ]:
# Compare a duração média das chamadas de cada plano para cada mês. 
#Crie um gráfico de barras para visualizar o resultado.

# Criando uma tabela que compare a duração média das chamadas de cada plano para cada mês
df_mean_calls = (df_calls_plan.pivot_table(index='call_month',
                                 columns='plan',
                                 values='duration_minutes',
                                 aggfunc='mean')
                                .reset_index()
                                .rename_axis(columns=None))

print('MÉDIA DA DURAÇÃO DAS CHAMADAS POR MÊS POR PLANO \n', df_mean_calls)

# Criando um gráfico
df_mean_calls.plot(x='call_month', kind='bar',
                  title='Média da duração das chamadas',
                  xlabel='Mês', ylabel='Tempo de duração (min)',
                  ylim=[0,9])

plt.show()


* a média da duração das chamadas é muito semelhante para os dois planos independentemente do mês do ano de 2018 que esteja sendo avaliado.

In [ ]:
# Compare o número de minutos que os usuários de cada plano necessitam a cada mês. 
# Construa um histograma.

# Criando df com somatório de minutos/mês consumido por usuário
df_minute_month_plan = (df_month_plan_used[['user_id', 'month_used','sum_duration_minutes', 'plan']].
                    rename(columns={'sum_duration_minutes': 'min_user_month'})
                       )
print('MINUTOS CONSUMIDOS POR USUÁRIO POR MÊS\n', df_minute_month_plan.head())

# Construindo o histograma
## Histograma do plano surf
df_minute_month_plan[df_minute_month_plan['plan'] == 'surf']['min_user_month'].plot(kind='hist', bins=30)

## Histograma do plano ultimate
df_minute_month_plan[df_minute_month_plan['plan'] == 'ultimate']['min_user_month'].plot(kind='hist', bins=30, alpha=0.8)

## Detalhamento do gráfico
plt.title('Minutos consumidos por usuário por mês')
plt.xlabel('Somatório de minutos por mês')
plt.ylabel('Frequência')
plt.legend(['Surf', 'Ultimate'])
plt.show()

* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição do somatório de minutos por mês é bastante semelhante e a maior frequência está em aproximadamente 400 minutos para os dois planos. 

In [ ]:
# Calcule a média e a variância da duração mensal das chamadas
print('Cálculo da duração das chamadas:\n')

# Cálculo para o plano surf
calls_surf = df_calls_plan[df_calls_plan['plan'] == 'surf']['duration_minutes']
mean_surf = calls_surf.mean().round(2)
print('A média para a duração da chamada no plano Surf é', mean_surf)
var_surf = np.var(calls_surf).round(2)
print('A variância para a duração da chamada no plano Surf é', var_surf)
cv_surf = calls_surf.std() / calls_surf.mean() * 100
print(f'O Coeficiente de Variação para a duração da chamada no plano surf é {cv_surf:.2f}%')

# Cálculo para o plano ultimate
calls_ultimate = df_calls_plan[df_calls_plan['plan'] == 'ultimate']['duration_minutes']
mean_ultimate = calls_ultimate.mean().round(2)
print('\nA média para a duração da chamada no plano Ultimate é', mean_ultimate)
var_ultimate = np.var(calls_ultimate).round(2)
print('A variância para a duração da chamada no plano Ultimate é', var_ultimate)
cv_ultimate = calls_ultimate.std() / calls_ultimate.mean() * 100
print(f'O Coeficiente de Variação para a duração da chamada no plano Ultimate é {cv_ultimate:.2f}%')



* existe pouca diferença entre a média, a variância e o coeficiente de variação para a duração das chamadas quando se compara os planos Surf e Ultimate
* Apesar de em termos absolutos a variância, para os dois planos, ter um valor baixo, o coeficiente de variação é maior do que 30% e forma desta pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.


In [ ]:
# Faça um diagrama de caixa para visualizar a distribuição da duração mensal das chamadas
plt.boxplot([calls_surf.dropna(), 
             calls_ultimate.dropna()], 
            labels=['Plano Surf', 'Plano Ultimate'])
plt.ylabel('Duração (min)')
plt.title('Comparação da distribuição dos tempos das chamadas')
plt.show()


* a média da duração das chamadas é muito semelhante para os dois planos independentemente do mês do ano de 2018 que esteja sendo avaliado.
* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição do somatório de minutos por mês é bastante semelhante e a maior frequência está em aproximadamente 400 minutos para os dois planos. 
* existe pouca diferença entre a média, a variância e o coeficiente de variação para a duração das chamadas quando se compara os planos Surf e Ultimate
* Apesar de em termos absolutos a variância, para os dois planos, ter um valor baixo, o coeficiente de variação é maior do que 30% e desta forma pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis, como pode ser comprovado pelo gráfico de caixa.

### Mensagens

In [ ]:
# Compare o número de mensagens que os usuários de cada plano costumam enviar a cada mês

# Criando um df para avaliar o consumo exclusivo de mensagem  
df_sms_month = df_month_plan_used[['user_id', 'month_used', 'number_messages', 'plan']]

print('MENSAGENS UTILIZADAS POR USUÁRIO POR MÊS\n', df_sms_month.head().head())
print()

# Construindo o histograma
## Histograma do plano surf
df_sms_month[df_sms_month['plan'] == 'surf']['number_messages'].plot(kind='hist', bins=10)

## Histograma do plano ultimate
df_sms_month[df_sms_month['plan'] == 'ultimate']['number_messages'].plot(kind='hist', bins=10, alpha=0.8)

## Detalhamento do gráfico
plt.title('Mensagens utilizadas por usuário por mês')
plt.xlabel('Somatório de mensagens por mês')
plt.ylabel('Frequência')
plt.legend(['surf', 'ultimate'])
plt.show()


* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição da quantidade de mensagens enviadas por mês é bastante semelhante e a maior frequência é um valor menor do que 25 mensagens para os dois planos. 

In [ ]:
#Calculando a média e a variância da quantidade de mensagens enviadas por mês

print('Cálculo para a quantidade de mensagens enviadas por mês:')

# Cálculo para o plano surf
sms_surf = df_sms_month[df_sms_month['plan'] == 'surf']['number_messages']

mean_sms_surf = sms_surf.mean().round(2)
print('A média de mensagens enviadas por mês para o plano surf é', mean_sms_surf)
var_sms_surf = np.var(sms_surf).round(2)
print('A variância para a quantidade de mensagens enviadas no plano surf é', var_sms_surf)
cv_sms_surf = sms_surf.std() / sms_surf.mean() * 100
print(f'O Coeficiente de Variação para a quantidade de mensagens enviadas por mês no plano Surf é {cv_sms_surf:.2f}%')

# Cálculo para o plano surf
sms_ultimate = df_sms_month[df_sms_month['plan'] == 'ultimate']['number_messages']

mean_sms_ultimate = sms_ultimate.mean().round(2)
print('\nA média de mensagens enviadas por mês para o plano ultimate é', mean_sms_ultimate)
var_sms_ultimate = np.var(sms_ultimate).round(2)
print('A variância para a quantidade de mensagens enviadas no plano ultimate é', var_sms_ultimate)
cv_sms_ultimate = sms_ultimate.std() / sms_ultimate.mean() * 100
print(f'O Coeficiente de Variação para a quantidade de mensagens enviadas por mês no plano Ultimate é {cv_sms_ultimate:.2f}%')

* Existe pouca diferença entre a média e a variância da duração das chamadas quando se compara os planos Surf e Ultimate
* Avaliando a variância para os dois planos, além de ter um valor absoluto alto, o coeficiente de variação é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.

In [ ]:
# Construindo um diagrama de caixa para visualizar a distribuição da quantidade de mensagens
plt.boxplot([sms_surf.dropna(), 
             sms_ultimate.dropna()], 
labels=['Plano Surf', 'Planon Ultimate'])
plt.ylabel('Quantidade de mensagens')
plt.title('Comparação da distribuição do envio mensal de mensagens')
plt.show()

* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição da quantidade de mensagens enviadas por mês é bastante semelhante, e a maior frequência é inferior a 25 mensagens em ambos os planos.
* Existe pouca diferença entre a média e a variância da duração das chamadas quando se comparam os planos Surf e Ultimate.
* Avaliando a variância para os dois planos, além de ter um valor absoluto alto, o coeficiente de variação é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.
* Considerando que o plano Surf tem apenas 50 mensagens incluídas, existe uma determinada quantidade de usuários deste plano que estão ultrapassando esta quantidade e pagando a mais por isso. Seria uma forma de abordagem da equipe de marketing sugerir a migração de plano para estes clientes.
* Nenhum cliente do plano ultimate atingiu sequer 25% do total de 1000 mensagens incluídas neste plano

### Internet

In [ ]:
# Compare a quantidade de tráfego de internet consumido pelos usuários por mês e por plano
# Criando um df para avaliar o consumo exclusivo de internet  
df_mb_month = df_month_plan_used[['user_id', 'month_used', 'sum_mb_used', 'plan']]

print('INTERNET UTILIZADAS POR USUÁRIO POR MÊS\n', df_mb_month.head())
print()

# Construindo o histograma
## Histograma do plano surf
df_mb_month[df_mb_month['plan'] == 'surf']['sum_mb_used'].plot(kind='hist', bins=30)

## Histograma do plano ultimate
df_mb_month[df_mb_month['plan'] == 'ultimate']['sum_mb_used'].plot(kind='hist', bins=30, alpha=0.8)

## Detalhamento do gráfico
plt.title('Quantidade de internet utilizada por usuário por mês')
plt.xlabel('Quantidade de dados consumida (mb)')
plt.ylabel('Frequência')
plt.legend(['Surf', 'Ultimate'])
plt.show()

* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição da quantidade de internet por mês é bastante semelhante, tendo sua maior frequência aproximadamente em 20.000mb em ambos os planos.
* Interessante pontuar que a quantidade de internet disponibilizada no plano Surf é de apenas 15.000mb, desta forma um parcela a ser estudada de usuários desse plano estão ultrapassando a quantidade de dados incluida neste plano.

In [ ]:
# Calculando a média e a variância da quantidade mensal de dados consumidos

print('Cálculo para a quantidade de dados consumidos por mês:\n')

# Cálculo para o plano Surf
mb_surf = df_mb_month[df_mb_month['plan'] == 'surf']['sum_mb_used']

mean_mb_surf = mb_surf.mean().round(2)
print('A média de dados consumidos por mês para clientes do plano surf é', mean_mb_surf, 'mb')
var_mb_surf = np.var(mb_surf).round(2)
print('A variância para a quantidade de dados consumidos por mês para clientes no plano surf é', var_mb_surf, 'mb')
cv_mb_surf = mb_surf.std() / mb_surf.mean() * 100
print(f'O Coeficiente de Variação para a quantidade mensal de dados consumidos no plano Surf é {cv_mb_surf:.2f}%')

# Cálculo para o plano Ultimate
mb_ultimate = df_mb_month[df_mb_month['plan'] == 'ultimate']['sum_mb_used']

mean_mb_ultimate = mb_ultimate.mean().round(2)
print('\nA média de dados consumidos por mês para clientes do plano ultimate é', mean_mb_ultimate, 'mb')
var_mb_ultimate = np.var(mb_ultimate).round(2)
print('A variância para a quantidade de dados consumidos por mês para clientes no plano ultimate é', var_mb_ultimate, 'mb')
cv_mb_ultimate = mb_ultimate.std() / mb_ultimate.mean() * 100
print(f'O Coeficiente de Variação para a quantidade mensal de dados consumidos no plano Ultimate é {cv_mb_ultimate:.2f}%')


* Existe pouca diferença entre a média e a variância da duração das chamadas quando se comparam os planos Surf e Ultimate.
* Ao avaliar a variância para os dois planos, além de ter um valor absoluto alto, o coeficiente de variação é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.
* Avaliando a variância para os dois planos, além de ter um valor absoluto alto, o coeficiente de variação é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.

In [ ]:
# Criando um diagrama de caixa para visualizar a distribuição da duração mensal das chamadas
plt.boxplot([mb_surf.dropna(), 
             mb_ultimate.dropna()], 
            labels=['Plano Surf', 'Plano Ultimate'])
plt.ylabel('Quantidade de dados (mb)')
plt.title('Comparação da distribuição do consumo mensal de dados')
plt.show()

* Apesar de o plano Surf ter um público maior que o plano Ultimate, a distribuição da quantidade de internet por mês é bastante semelhante, tendo sua maior frequência aproximadamente em 20.000mb em ambos os planos.
* Interessante pontuar que a quantidade de internet disponibilizada no plano Surf é de apenas 15.000mb, desta forma um parcela a ser estudada de usuários desse plano estão ultrapassando a quantidade de dados incluida neste plano.
* Existe pouca diferença entre a média e a variância da duração das chamadas quando se comparam os planos Surf e Ultimate.
* Ao avaliar a variância para os dois planos, além de ter um valor absoluto alto, ela é mais de 357 vezes maior que a média. Pode-se então considerar um enorme espalhamento dos dados
* Avaliando a variância para os dois planos, além de ter um valor absoluto alto, o coeficiente de variação é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis.


## Receita

[Da mesma forma que você estudou o comportamento dos usuários, descreva estatisticamente as receitas dos planos.]

In [ ]:
# Compare os dados estatísticos básicos para o consumo real por mês por plano
plan_revenue_stats = df_month_plan_used.groupby('plan')['monthly_revenue'].describe()
print(plan_revenue_stats)

* A média de receita para o plano Surf é de 57,29usd
* A média da receita para o plano Ultimate é de 72,11usd 
* Em mais de 50% dos dados dos usuários do plano Surf o valor do consumo mensal está acima de 46 usd
* Ao menos 75% dos dados do usuário do plano Ultimate o valor do consumo mensal foi apenas o pacote básico de 70 usd

In [ ]:
# Compare a receita média de cada plano para cada mês. 
# Crie um gráfico de barras para visualizar o resultado

# Criando uma tabela que compare a receita média de cada plano para cada mês
mean_revenue = (df_month_plan_used.pivot_table(index='month_used', 
                                               columns='plan',
                                               values='monthly_revenue',
                                               aggfunc='mean').
    reset_index().
    rename_axis(columns=None)
    )

print('MÉDIA DA RECEITA POR MÊS POR PLANO \n', mean_revenue)
print()

# Criando um gráfico
mean_revenue.plot(x='month_used', kind='bar',
                  title='Média da receita de cada plano para cada mês',
                  xlabel='Mês', ylabel='Receita (usd)', ylim=[0,90])

plt.show()

* Os usuários do plano Ultimate consomem muito pouco além do pacote contratado
* Os usuários do plano Surf na maioria dos meses consomem mais do que o doblo do pacote básico de 20 usd.

In [ ]:
# Calculando a variância e o coeficiente de variação da receita mensal real

print('Cálculo para a receita mensal real:\n')

# Cálculo para o plano Surf
revenue_surf = df_month_plan_used[df_month_plan_used['plan'] == 'surf']['monthly_revenue']

var_revenue_surf = np.var(revenue_surf).round(2)
print('A variância para a receita mensal para clientes no plano surf é', var_revenue_surf, 'usd')
cv_revenue_surf = revenue_surf.std() / revenue_surf.mean() * 100
print(f'O Coeficiente de Variação para a receita mensal no plano Surf é {cv_revenue_surf:.2f}%')

# Cálculo para o plano Ultimate
revenue_ultimate = df_month_plan_used[df_month_plan_used['plan'] == 'ultimate']['monthly_revenue']

var_revenue_ultimate = np.var(revenue_ultimate).round(2)
print('\nA variância para a receita mensal para clientes no plano Ultimate é', var_revenue_ultimate, 'usd')
cv_revenue_ultimate = revenue_ultimate.std() / revenue_ultimate.mean() * 100
print(f'O Coeficiente de Variação para a receita mensal no plano Ultimate é {cv_revenue_ultimate:.2f}%')


* Ao avaliar a variância para os dois planos, o valor absoluto alto. Porém, o coeficiente de variação para o plano Surf é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis. No entanto, o coeficiente de variação para o Ultimate é menor que 15% e desta forma, pode-se considerar baixa dispersão na amostra com dados homogêneos e altamente consistentes.

In [ ]:
# Construa um diagrama de caixa para visualizar a distribuição da duração mensal das chamadas
plt.boxplot([revenue_surf.dropna(),
             revenue_ultimate.dropna()], 
            labels=['Plano Surf', 'Plano Ultimate'])
plt.ylabel('Receita (usd)')
plt.title('Comparação da distribuição das receitas')
plt.show()


* A média de receita para o plano Surf é de 57,29usd
* A média da receita para o plano Ultimate é de 72,11usd 
* Em mais de 50% dos dados dos usuários do plano Surf o valor do consumo mensal está acima de 46 usd
* Ao menos 75% dos dados do usuário do plano Ultimate o valor do consumo mensal foi apenas o pacote básico de 70 usd
* Os usuários do plano Ultimate consomem muito pouco além do pacote contratado
* Os usuários do plano Surf na maioria dos meses consomem mais do que o doblo do pacote básico de 20 usd.
* Ao avaliar a variância para os dois planos, o valor absoluto alto. Porém, o coeficiente de variação para o plano Surf é maior do que 30% e desta forma, pode-se considerar alta dispersão na amostra com dados muito heterogêneos e instáveis. No entanto, o coeficiente de variação para o Ultimate é menor que 15% e desta forma, pode-se considerar baixa dispersão na amostra com dados homogêneos e altamente consistentes.

## Teste hipóteses estatísticas

#### Teste da hipótese de que a receita média dos usuários dos planos Ultimate e Surf são diferentes.

Parâmetros:

H0 = A receita média dos usuários dos planos Ultimate e Surf é igual.
H1 = A receita média dos usuários dos planos Ultimate e Surf é diferente.

O teste estatístico selecionado foi o scipy.stats.ttest_ind, tendo em vista que buscava-se testar a hipótese de média de duas populações estatísticas.

O valor de alpha foi de 5%

Utilizou-se um teste bicaudal, uma vez que está sendo buscada a 'diferença' e não se se um é maior ou menor que o outro.


In [ ]:
# Teste as hipóteses
# Previously prepared samples
revenue_surf
revenue_ultimate

# Setting the parameters
alpha = 0.05                  # critical level of statistical significance


# Running the tests
results = st.ttest_ind(revenue_surf, revenue_ultimate)

# Extract the p-value from the results
print('valor-p:', results.pvalue)

# comparing the p-value with the threshold
if results.pvalue < alpha:
    print('Rejeitamos H0: as médias de receita são estatisticamente diferentes')
else:
    print('Não rejeitamos H0: não há evidência suficiente de diferença')

    

#### Teste da hipótese de que a receita média dos usuários da área de NY-NJ difere dos usuários das demais regiões.

Parâmetros:

H0 = A receita média dos usuários da área NY-NJ e dos usuários das demais regiões é igual.

H1 = A receita média dos usuários da área NY-NJ e dos usuários das demais regiões  é diferente.

O teste estatístico selecionado foi o scipy.stats.ttest_ind, tendo em vista que buscava-se testar a hipótese de média de duas populações estatísticas.

O valor de alpha foi de 5%

Utilizou-se um teste bicaudal, uma vez que está sendo buscada a 'diferença' e não se um é maior ou menor que o outro.


In [ ]:
# Preparing the samples

# Creating a df with the revenues and the areas
users_area = (df_month_plan_used.merge(df_users[['user_id','state_msa']],
                                      on='user_id', how='left'))


In [ ]:
# Testing the hypothesis

# Creating the samples
ny_nj = users_area[users_area['state_msa'] == 'NY-NJ-PA']['monthly_revenue']
other_areas = users_area[~(users_area['state_msa'] == 'NY-NJ-PA')]['monthly_revenue']

# Setting the parameters
alpha = 0.05                  # critical level of statistical significance

# Running the tests
results = st.ttest_ind(ny_nj, other_areas)

# Extract the p-value from the results
print('valor-p:', results.pvalue)

# comparing the p-value with the threshold
if results.pvalue < alpha:
    print('Rejeitamos H0: as médias de receita são estatisticamente diferentes')
else:
    print('Não rejeitamos H0: não há evidência suficiente de diferença')


## Conclusão geral

Avaliando o comportamento de consumo da amostra estudada, pode-se notar que a distribuição dos serviços utilizados, tanto no somatório de minutos utilizados como no envio de mensagens e no consumo de dados por mês, é bastante semelhante e as maiores frequências de utilização são relativamente próximas para ambos os planos.

Comparando os dados dos dois planos nesta amostra, é notado que a média foi sempre semelhante para o mesmo serviço avaliado e o valor da variância foi na maioria das vezes grande, revelando uma alta dispersão na amostra com dados muito heterogêneos e instáveis, como pode ser comprovado pelos gráficos de caixa.

Em função destes dados de média e variância semelhantes nos dois planos, para a realização do teste das hipóteses estatísticas utilizando a função scipy.stats.ttest_ind o parâmetro equal_var foi considerado True por padrão.

Apesar do consumo dos serviços ser semelhante, a escolha do plano pelos clientes foi diferente, o que reflete em um comportamento discrepante quando se analisa a receita. O pacote básico do plano Surf era 20 usd, mas a média de receita  foi de 57,29 usd. Já para o plano Ultimate o básico era 70 usd e a média foi 72,11 usd. Isto revela que, por terem o comportamento de consumo semelhante, os usuários do plano Surf normalmente precisavam utilizar recursos adicionais que encareciam o valor no fim do mês.

Apesar do consumo acima do pacote básico do plano, o valor pago não atingia o valor do plano Ultimate e por conta disso os clientes não devem se sentir estimulados a fazer a migração. Avaliar com o setor comercial a criação de um plano intermediário para garantisse uma receita mensal maior independente do consumo do pacote inteiro de serviços.